In [1]:
from database.MongoDBClient import MongoDBClient
from database.Recipe import Recipe
from transformers import AutoTokenizer
from model.model import Chefformer
from config.ModelSettings import ModelSettings
import torch

/Users/michalekb/Documents/GitHub_Personal/chefformer_repo/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/michalekb/Documents/GitHub_Personal/chefformer_repo/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ModelSettings()

ModelSettings(name='Chefformer', vocab_size=50257, embedding_size=768, max_context_length=512, num_attn_heads=1, dropout_prob=0.1, num_layers=12)

In [3]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
chefformer = Chefformer(tokenizer)


In [4]:
s = """Title: Italian Cream Cake

Ingredients:
- Buttermilk (1 cup)
- Baking soda (1 teaspoon)
- Butter (1/2 cup)
- Shortening (1/2 cup)
- White sugar (2 cups)
- Eggs (5)
- Vanilla extract (1 teaspoon)
- Flaked coconut (1 cup)
- Baking powder (1 teaspoon)
- All-purpose flour (2 cups)
- Cream cheese (8 ounces)
- Butter (1/2 cup)
- Vanilla extract (1 teaspoon)
- Confectioners' sugar (4 cups)
- Light cream (2 tablespoons)
- Chopped walnuts (1/2 cup)
- Sweetened flaked coconut (1 cup)
...

Instructions: Finely grate the zest from 4 of the lemons, then pulse in a mini food processor with 3 tablespoons salt. Squeeze the juice from 2 of the zested lemons into a large bowl; add the lemongrass, olive oil, Sriracha, garlic, fish sauce, brown sugar and 1 teaspoon of the lemon salt. Thinly slice the 2 remaining lemons and add to the bowl.Using kitchen shears, cut through the shrimp shells along the outer curve, leaving the shells on. Remove the veins and rinse the shrimp. Add to the bowl with the lemongrass marinade and toss; cover and refrigerate for 1 hour. Meanwhile, soak 12 to 15 wooden skewers in water.Preheat a grill to medium high. Thread the shrimp onto the skewers. Brush the grill with vegetable oil, then add the lemon slices and grill until they begin to char, turning once, about 2 minutes; transfer to a platter. Stir the mint into the remaining marinade. Add the shrimp skewers to the grill and cook, brushing with the marinade-mint mixture, until the shells begin to char, 2 to 3 minutes per side. Transfer to the platter and sprinkle with the lemon salt to taste."""


s = 'hey how are you'

s = ['hey how', 'are you doing today']

In [5]:
x = tokenizer(s, return_tensors='pt', padding=True, truncation=True, max_length=512)
x['input_ids']

tensor([[20342,   703, 50256, 50256],
        [  533,   345,  1804,  1909]])

In [6]:
embeddings = chefformer.get_embeddings(s)
print(embeddings)
print(embeddings.shape)

tensor([[[-0.0000,  0.0000,  0.1138,  ..., -0.1554, -0.0000, -0.5149],
         [-0.2089, -1.8000,  1.9777,  ...,  0.0000, -1.6095, -2.0233],
         [-0.5934,  0.2361,  0.1434,  ..., -1.3299,  0.0306,  2.1270],
         [ 1.2849, -0.7941, -0.3163,  ..., -1.7543,  0.7120,  2.5198]],

        [[-0.2969,  1.0793, -0.7161,  ...,  1.0142, -1.2139, -1.3217],
         [-1.8078, -1.0926, -0.0000,  ...,  0.1517, -0.4934,  1.3260],
         [-1.3785, -2.2214,  1.2223,  ..., -1.4352,  0.0000,  2.2610],
         [ 1.1545, -0.7550, -0.7543,  ..., -1.7684,  1.6167,  1.1717]]],
       grad_fn=<MulBackward0>)
torch.Size([2, 4, 768])


In [7]:
# Test parts of attention

def scaled_dot_product_attention(query, key, value):
    batch_size, seq_length, d_attn_head = query.shape[0], query.shape[1], query.shape[-1]
    attenion_weights = torch.bmm(query, torch.transpose(key, 1, 2)) # Q * K.T... (batch_size, seq_len, seq_len)
    attenion_weights = attenion_weights / torch.sqrt(torch.tensor(d_attn_head, dtype=torch.float32)) # Scale the scores

    mask = torch.tril(torch.ones(batch_size, seq_length, seq_length, dtype=torch.int)) # Gather attention matrix... (batch_size, seq_len, seq_len)
    attenion_weights[mask == 0] = float('-inf') # Set scores to -inf where tokens are masked

    attenion_weights = torch.softmax(attenion_weights, dim=-1) # Take softmax along rows (each token's row should sum to 1)
    return torch.bmm(attenion_weights, value)


example_embeddings = torch.tensor([[[0.5, 0.5, 0.5],
                           [-0.1, -0.1, -0.1]],

                           [[0.3, 0.3, 0.3],
                            [1.0, 1.0, 1.0]]])

print(scaled_dot_product_attention(example_embeddings, example_embeddings, example_embeddings))
print()


attn_output = chefformer.test_attention_head(['hey how are you', 'today'])
print(attn_output)
print(attn_output.shape)

tensor([[[0.5000, 0.5000, 0.5000],
         [0.1844, 0.1844, 0.1844]],

        [[0.3000, 0.3000, 0.3000],
         [0.8395, 0.8395, 0.8395]]])

tensor([[[-0.5148,  1.7873, -0.0142,  ..., -0.4701,  0.0809,  0.3720],
         [-0.4099,  1.4756, -0.1996,  ..., -0.5630,  0.1435,  0.4865],
         [ 0.1994,  0.9074, -0.3376,  ..., -0.0452, -0.1096,  0.3938],
         [ 0.1296,  0.5073, -0.6625,  ..., -0.0698, -0.2126, -0.2522]],

        [[-2.0693,  2.4700, -0.1122,  ...,  0.1081, -0.6427,  0.6675],
         [-1.7935,  2.1422, -0.0645,  ...,  0.1247, -0.5704,  0.3258],
         [-1.1114,  1.6720,  0.1055,  ...,  0.3995, -0.6320,  0.0312],
         [-0.0982,  0.4427, -0.2004,  ...,  0.1335, -0.5454, -1.0130]]],
       grad_fn=<BmmBackward0>)
torch.Size([2, 4, 768])


In [8]:
# Test multi head attention
attn_output = chefformer.test_multihead_masked_self_attention(['hey how are you', 'today'])
print(attn_output)
print(attn_output.shape)

tensor([[[-0.0825,  0.2461,  0.2487,  ..., -0.1740, -0.3485, -0.3304],
         [-0.0477,  0.5380,  0.3895,  ..., -0.3798, -0.1227, -0.4228],
         [-0.0277,  0.2833,  0.6118,  ..., -0.0224, -0.1902, -0.2782],
         [ 0.0669,  0.6666,  0.5990,  ..., -0.3259,  0.0475, -0.3448]],

        [[-0.2912,  0.4590,  1.0686,  ...,  0.3038, -0.3950, -0.5570],
         [-0.1959,  0.5141,  0.3011,  ...,  0.4698, -0.3285, -0.4241],
         [-0.2252,  0.2966,  0.6457,  ...,  0.2550, -0.3049, -0.5717],
         [-0.2752, -0.0362,  0.5748,  ...,  0.1642, -0.2457, -0.2835]]],
       grad_fn=<ViewBackward0>)
torch.Size([2, 4, 768])


In [9]:
# Test feed foward network
ff_output = chefformer.test_feed_forward(['hey how are you', 'today'])
print(ff_output)
print(ff_output.shape)

tensor([[[-0.0000,  0.0597,  0.0132,  ...,  0.1493, -0.1238, -0.0011],
         [-0.0229,  0.1023, -0.0099,  ...,  0.0404,  0.0043,  0.0524],
         [-0.0118,  0.0022, -0.0227,  ...,  0.0103,  0.0905,  0.0426],
         [-0.0411, -0.0196, -0.0504,  ...,  0.0621,  0.0767,  0.0930]],

        [[-0.1224,  0.0741,  0.0027,  ...,  0.1775, -0.0424,  0.1049],
         [-0.0511,  0.1178,  0.0000,  ...,  0.1215,  0.0000, -0.0223],
         [-0.0253,  0.0565, -0.0190,  ...,  0.1089,  0.0950, -0.0682],
         [-0.1208,  0.1070, -0.0401,  ...,  0.0000,  0.0298, -0.0005]]],
       grad_fn=<MulBackward0>)
torch.Size([2, 4, 768])


In [10]:
# Test decoder block
decoder_block_output = chefformer.test_decoder_block(['hey', 'how are you'])
print(decoder_block_output)
print(decoder_block_output.shape)

tensor([[[-3.5148e-01,  1.8826e+00,  2.1472e-01,  ..., -2.8326e-01,
          -1.3037e+00, -1.1113e+00],
         [-1.9035e+00, -6.2239e-01, -0.0000e+00,  ...,  1.0807e+00,
          -1.6520e+00,  2.1043e+00],
         [-4.5309e-01, -3.2025e-01,  2.1203e-01,  ..., -1.4564e+00,
          -4.3855e-01,  2.9571e+00]],

        [[-3.1618e+00, -0.0000e+00,  9.5521e-01,  ..., -4.4386e-01,
           0.0000e+00, -4.9654e+00],
         [-0.0000e+00, -3.3619e+00, -2.8209e-01,  ..., -0.0000e+00,
           1.6741e-01,  8.9319e-01],
         [-7.1876e-01, -7.1331e-01, -3.2612e-03,  ..., -2.7703e+00,
           0.0000e+00,  0.0000e+00]]], grad_fn=<MulBackward0>)
torch.Size([2, 3, 768])


In [11]:
# Test entire chefformer
chefformer_output = chefformer(['hey', 'how are you'])
print(chefformer_output)
print(chefformer_output.shape)

tensor([[[-0.2864,  0.0706, -0.0516,  ..., -1.1524, -0.0228,  0.4122],
         [ 0.1687, -0.4520,  0.5003,  ..., -0.2661, -0.9080,  1.1050],
         [-0.0053, -0.0519,  0.5761,  ...,  0.4606,  0.0649, -0.2990]],

        [[-0.6815,  0.5057,  0.4259,  ...,  0.3055,  1.0625, -0.4669],
         [-1.2500,  0.0334, -0.9940,  ...,  0.0336, -0.7009,  0.8192],
         [-0.8348,  0.0420, -0.6254,  ..., -0.1874, -0.8917, -0.2654]]],
       grad_fn=<ViewBackward0>)
torch.Size([2, 3, 50257])


In [12]:
chefformer.parameters

<bound method Module.parameters of Chefformer(
  (Embeddings): Embeddings(
    (token_embeddings): Embedding(50257, 768)
    (position_embeddings): Embedding(512, 768)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (DecoderBlocks): ModuleList(
    (0-11): 12 x DecoderBlock(
      (MultiHeadMaskedSelfAttention): MultiHeadMaskedSelfAttention(
        (attn_heads): ModuleList(
          (0): AttentionHead(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (output_projection): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (PositionWiseFeedForward): PositionWiseFeedForward(
        (f1): Linear(in_features=768, out_features=3072, bias=True)
        (f2): Linear(in_features=3072, out_features=768, bias=True)
        (gelu): GELU(app